# 1.1 Visualizing OPM MEG Epochs with MNE-Python

This notebook walks through the major ways to visualize epoched MEG data using MNE-Python,
adapted from the [MNE Visualizing Epoched Data tutorial](https://mne.tools/stable/auto_tutorials/epochs/20_visualize_epochs.html).

Instead of the sample SQUID-MEG dataset, we use real OPM MEG data from our passive auditory
oddball paradigm (sub-001, ses-01). The preprocessed clean epochs file
(`proc-clean_epo.fif`) is the starting point — it has already been filtered,
HFC-denoised, ICA-cleaned, and bad epochs rejected.

---
### Paradigm design

| Parameter | Value |
|-----------|-------|
| Standard tone | 1000 Hz |
| Deviant tone | 1200 Hz |
| Tone duration | 400 ms (50 ms rise/fall ramps) |
| Total trials | 1000 (~84% standard, ~16% deviant) |
| Deviant count | ~160, separated by 3–7 standards (mean gap 5.1 ± 1.3) |
| ISI | 700–900 ms jittered (~25 discrete bins of ~8.33 ms) |
| Mean trial duration | 1200 ms (400 ms tone + 800 ms mean ISI) |
| Onset-to-onset range | 1100–1300 ms |
| Total recording | ~24 min |
| Task | Passive fixation (central fixation cross) |
| Opening sequence | First 10 trials are all standards (establish regularity) |

**Blink trials:** One blink opportunity follows each deviant (~162 total). A green fixation
cross appears for 1.5 s. The blink cue onset is at least 1100 ms after deviant onset —
well after the MMN window (100–250 ms post-deviant) — so blink artifacts do not
contaminate the MMN response of interest.

**Conditions in this epochs file:**
- `standard_onset` — frequent 1000 Hz tones (~840 trials before rejection)
- `deviant_onset` — rare 1200 Hz tones (~160 trials before rejection), the MMN-eliciting stimuli

---
### Running this notebook in the mne-opm environment

This notebook is designed to run inside the `mne-opm` project environment. Launch it with:

```bash
uv run --project $MNE_OPM_DIR jupyter lab
# or
uv run --project $MNE_OPM_DIR jupyter notebook
```

Once open, make sure the kernel matches the mne-opm Python interpreter (it should select it automatically if launched this way).

For **interactive** plot windows (scrollable epochs browser), run cells from a terminal using:
```bash
uv run --project $MNE_OPM_DIR python -c "..."
```
or use `%matplotlib qt` instead of `%matplotlib inline` below. Interactive mode is recommended for `epochs.plot()` since it allows you to scroll, mark bad epochs, and zoom.

## 0. Setup

In [ ]:
# Use inline static figures for notebook rendering.
# Switch to %matplotlib qt for interactive scrollable plots (e.g. epochs.plot())
%matplotlib inline
#%matplotlib qt

import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level('WARNING')  # reduce console noise
print(f'MNE version: {mne.__version__}')

In [ ]:
# ─── USER SETTINGS ───────────────────────────────────────────────────────────
# Set this to the folder containing your preprocessed output files.
# All files listed here are expected to live in the same directory.

DERIV_DIR = pathlib.Path('/path/to/bids/derivatives/analysis1__/sub-001/ses-01/meg')

SUBJECT   = 'sub-001'
SESSION   = 'ses-01'
TASK      = 'oddball'

# Epoch condition labels (must match what was used during epoching)
STANDARD_COND = 'standard_onset'
DEVIANT_COND  = 'deviant_onset'
# ─────────────────────────────────────────────────────────────────────────────

# Build the file path using BIDS-style naming
epo_file = DERIV_DIR / f'{SUBJECT}_{SESSION}_task-{TASK}_proc-clean_epo.fif'

print(f'Epochs file: {epo_file}')
print(f'File exists: {epo_file.exists()}')

## 1. Load the Epochs

We load the already-cleaned epochs. `preload=True` brings all the data into RAM upfront,
which is required for most visualization and analysis operations.

In [ ]:
epochs = mne.read_epochs(epo_file, preload=True)
print(epochs)

In [ ]:
# Quick sanity check: how many epochs per condition?
# Expected before rejection: ~840 standards, ~160 deviants.
# After ICA + artifact rejection the actual counts will be somewhat lower.
for cond in epochs.event_id:
    n = len(epochs[cond])
    print(f'  {cond}: {n} epochs')

print(f'\nChannel types: {epochs.get_channel_types()[:3]}... ({len(epochs.ch_names)} total channels)')
print(f'Epoch window:  {epochs.tmin:.3f} s → {epochs.tmax:.3f} s')
print(f'Sampling rate: {epochs.info["sfreq"]} Hz')

## 2. Visualize Epochs as Time Series — Deviant Trials Only

The most direct way to inspect epochs is `epochs.plot()`, which opens an interactive
browser where you can scroll through trials, zoom in/out, and flag bad epochs.

Here we plot **only the deviant trials** by indexing the epochs object with the condition label.
There are only ~160 deviant trials (16% of 1000 total), separated from one another by
3–7 standard tones. Because they are rare, the brain's prediction-error response
(the **Mismatch Negativity, MMN**) is elicited at each one — making it worth inspecting
these trials closely for artifacts before averaging.

> **About blink trials:** A blink cue (green fixation cross, 1.5 s) follows each deviant at
> least 1100 ms after onset. Those blink-period epochs are *not* included in `deviant_onset` —
> they were triggered separately. So what you see here are clean stimulus-locked epochs
> that should be free of blink contamination within the MMN window.

**Interactive keyboard shortcuts** (best used in `%matplotlib qt` mode):
| Key | Action |
|-----|--------|
| `←` / `→` | Scroll epochs |
| `↑` / `↓` | Scroll channels |
| `+` / `-` | Zoom vertical scale |
| `b` | Toggle butterfly mode |
| Left-click epoch | Mark as bad |
| Left-click channel name | Mark channel as bad |

In [ ]:
# Plot only the deviant condition epochs.
# In inline mode this shows a static snapshot; switch to %matplotlib qt for the
# full interactive browser.

fig = epochs[DEVIANT_COND].plot(
    n_epochs=5,      # show 5 epochs at a time
    n_channels=20,   # show 20 channels at a time
    title=f'Deviant epochs ({DEVIANT_COND})',
    show=True,
    block=False,     # don't block execution (set True in interactive Qt mode)
)
plt.tight_layout()

## 3. Butterfly Mode — All Sensors Simultaneously

In a **butterfly plot** all channels of the same type are overlaid on a single axes,
making it easy to see the global waveform shape and spot outlier channels.

For OPM data, grouping by **sensor axis (X, Y, Z)** is more informative than `group_by='type'`
(which lumps everything together) or `group_by='position'` (which uses k-means clustering
and can produce empty groups). Each OPM slot records up to three orthogonal field components;
separating them into three butterfly panels lets you compare signal quality and artifact
character across axes — useful because movement artifacts often affect axes differently.

We build the grouping dict manually by parsing channel names for their axis suffix
(`[X]`, `[Y]`, `[Z]`). This gives **3 windows** — one per axis across all epochs.

> **What to look for:** Channels that sit far outside the pack (persistent outliers) are
> candidates for bad-channel marking. Large-amplitude deflections shared across many
> channels within one axis panel suggest axis-specific noise (e.g. a poorly fitted sensor).
> Correlated deflections across all three panels suggest a common-mode artifact (movement).


In [ ]:
# Build a group_by dict that assigns each channel to its axis (X, Y, or Z).
# Channel names follow the pattern: LOCATION SERIAL AXIS  (e.g. 'Iz 1T Z', 'O10 1U X')
# BNC trigger channels (also ending in ' Z') are excluded by name prefix.

axis_groups = {'X': [], 'Y': [], 'Z': []}
ch_names = epochs.ch_names

for idx, ch in enumerate(ch_names):
    if ch.startswith('BNC'):
        continue  # skip trigger/BNC channels
    if ch.endswith(' X'):
        axis_groups['X'].append(idx)
    elif ch.endswith(' Y'):
        axis_groups['Y'].append(idx)
    elif ch.endswith(' Z'):
        axis_groups['Z'].append(idx)

# Remove any empty axes (e.g. if only radial Z channels were recorded)
axis_groups = {k: v for k, v in axis_groups.items() if v}

for ax, idxs in axis_groups.items():
    print(f'  {ax}: {len(idxs)} channels')

if not axis_groups:
    print('WARNING: No axis suffixes detected in channel names.')
    print('Sample names:', ch_names[:5])
    print('Falling back to group_by="type"')

In [ ]:
# Why .copy().pick('mag') instead of just using picks=?
# --------------------------------------------------------
# You might expect that passing picks=mag_indices to .plot() would be enough
# to show only magnetometer channels. It does filter *what is plotted*, but
# group_by='type' draws one row-label per channel type present in the epochs
# *object* — not just the ones selected by picks. So even if you only plot
# mag channels, the browser still renders empty rows for EOG, stim, eyetracking,
# pupil, and reference channels.
#
# The fix is to strip those other types out of the epochs object itself first
# using .pick('mag'). We use .copy() so the original epochs (with all channel
# types intact) is preserved for the other sections of this notebook.

epochs_mag = epochs.copy().pick('mag')

# Rebuild the axis index lists on the reduced channel set.
# After .pick('mag') the channel indices are renumbered from 0, so we can't
# reuse the axis_groups built earlier (those indices pointed into the full
# channel list). We rebuild using the same suffix logic as before.
axis_groups_mag = {'X': [], 'Y': [], 'Z': []}
for idx, ch in enumerate(epochs_mag.ch_names):
    if ch.endswith(' X'):
        axis_groups_mag['X'].append(idx)
    elif ch.endswith(' Y'):
        axis_groups_mag['Y'].append(idx)
    elif ch.endswith(' Z'):
        axis_groups_mag['Z'].append(idx)

# Gray lines in the butterfly plot are bad channels — MNE renders them in gray
# to distinguish them from good channels (shown in blue). They were flagged
# during preprocessing and stored in epochs.info['bads']. Run the line below
# to see which channels are marked bad:
print('Bad channels:', epochs_mag.info['bads'])

# Plot one butterfly window per axis — 3 windows total (all epochs, both conditions).
for axis, idxs in axis_groups_mag.items():
    epochs_mag.plot(
        picks=idxs,
        butterfly=True,
        group_by='type',
        title=f'Butterfly — {axis} axis ({len(idxs)} mag channels)',
        show=True,
        block=False,
    )


## 4. Sensor Layout — `plot_sensors()`

`epochs.plot_sensors()` draws sensor positions onto a head diagram. Two views are shown:

- **2D projection** (`kind='topomap'`, default) — flat overhead view useful for confirming layout at a glance
- **3D positions** (`kind='3d'`) — interactive rotatable view that reveals the helmet geometry and sensor depth; particularly useful for OPM arrays where sensors are form-fitted to the head

Both are a quick sanity check that:
- The digitisation / sensor geometry loaded correctly
- No sensors are in implausible locations
- Bad channels (if any) are visible as distinct markers

With `show_names=True` you can identify individual channel labels, which is helpful
when you want to pick a specific channel for targeted analysis.


In [ ]:
# With channel names labelled — useful when you need to identify a specific sensor
fig = epochs.plot_sensors(show_names=True)
fig.suptitle('OPM Sensor Layout (labelled)', fontsize=12)

In [ ]:
# 3D sensor positions — useful for OPM arrays where helmet geometry matters
# kind='3d' opens an interactive matplotlib 3D axes you can rotate
epochs.plot_sensors(kind='3d', show_names=False)


## 5. Power Spectral Density of Epochs

Before diving into time-domain waveforms it's worth looking at the **frequency content**
of the data. `epochs.compute_psd()` returns an `EpochsSpectrum` object. Calling `.plot()`
on it shows the mean PSD (with confidence interval) across epochs and channels.

For our OPM oddball data, key landmarks to check:
- **1/f slope** — MEG signals naturally have more power at low frequencies
- **60 Hz (and 120/180 Hz harmonics)** — power-line noise. These should have been
  notch-filtered during preprocessing, so a clean spectrum here is a good sign.
- **Alpha (~10 Hz)** — spontaneous occipital oscillations; passive task with fixation
  means alpha should be present and stable across conditions
- **Noise floor above ~100 Hz** — OPM sensors have a white-noise floor; the PSD should
  flatten out above the signal band
- **Low-frequency drift (<1 Hz)** — movement artifacts in OPM data tend to appear
  at very low frequencies; the epoch window and baseline period influence this

> **Note:** `compute_psd()` replaces the older `plot_psd()` method in MNE ≥ 1.2.

In [ ]:
# A note on bad channels and explicit picks:
# MNE excludes bad channels by default only when picks=None. As soon as you
# pass an explicit picks argument (e.g. picks='mag'), bad channels that fall
# within that selection are included. We therefore pass exclude='bads' to
# compute_psd() so that flagged channels are consistently excluded here and
# in the overlay and topomap cells that follow.

spectrum_deviant = epochs[DEVIANT_COND].compute_psd(
    method='welch',
    fmin=1.0,
    fmax=100.0,
    n_fft=512,
    n_overlap=256,
    exclude='bads',   # exclude channels marked bad during preprocessing
)

# .plot() shows mean PSD ± CI across epochs, one line per channel.
# show=False suppresses MNE's internal plt.show() call, which triggers a
# harmless but noisy UserWarning in inline/notebook backends.
# MNE's PSD figure uses a custom axes layout that is incompatible with
# tight_layout(), so we skip that call here — the figure renders correctly without it.
fig = spectrum_deviant.plot(
    picks='mag',        # OPM sensors are mag-type
    amplitude=False,    # plot power (T²/Hz) not amplitude (T/√Hz)
    dB=True,            # log scale: easier to see broadband structure
    show=False,
)
fig.suptitle(f'PSD — {DEVIANT_COND}', fontsize=12)
plt.show()


In [ ]:
# Overlay both conditions to compare spectral profiles.
#
# Note on trial counts: compute_psd() calculates a spectrum for each epoch
# individually and then averages them, so every trial contributes equally to
# the mean. The difference in trial counts (~840 standard vs ~160 deviant)
# does NOT bias the mean PSD — a condition with 10x more trials gives the
# same mean PSD as one with fewer trials if the underlying signal is the same.
#
# You may notice the standard condition has higher overall power than deviant.
# This is likely a genuine neural effect rather than an artifact. Possible explanations:
#
#   1. Deviant-related oscillatory suppression — the MMN response involves
#      event-related desynchronization (ERD), a reduction of ongoing oscillations
#      following a prediction error. This would reduce average power in deviant epochs.
#
#   2. Post-deviant blink context — every deviant is immediately followed by a
#      blink trial (green fixation cross). The neural state entering the post-deviant
#      period is therefore slightly different from the steady stream of standards,
#      which may suppress ongoing oscillatory activity.
#
#   3. Auditory cortex entrainment — standards are heard repeatedly in close
#      succession (3–7 in a row before each deviant), which can drive stronger
#      steady-state entrainment of auditory cortex and therefore higher baseline
#      oscillatory power in those epochs compared to the less-frequent deviants.

spectrum_standard = epochs[STANDARD_COND].compute_psd(
    method='welch', fmin=1.0, fmax=100.0, n_fft=512, n_overlap=256,
    exclude='bads',   # exclude bad channels, consistent with spectrum_deviant above
)

fig, ax = plt.subplots(figsize=(8, 4))

for spec, label, color in [
    (spectrum_standard, STANDARD_COND, 'steelblue'),
    (spectrum_deviant,  DEVIANT_COND,  'tomato'),
]:
    psds, freqs = spec.get_data(return_freqs=True)  # shape: (n_epochs, n_ch, n_freqs)
    mean_psd = psds.mean(axis=(0, 1))               # average over epochs and channels
    ax.semilogy(freqs, mean_psd, label=label, color=color, linewidth=1.5)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Power (T²/Hz)')
ax.set_title('Mean PSD: Standard vs Deviant')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()


## 6. Spectral Power as Scalp Topography — `plot_topomap()`

`EpochsSpectrum.plot_topomap()` shows **where on the scalp** the power lives within
each canonical frequency band. By default it plots five bands:

| Band | Range |
|------|-------|
| δ (delta) | 1–4 Hz |
| θ (theta) | 4–8 Hz |
| α (alpha) | 8–12 Hz |
| β (beta) | 12–30 Hz |
| γ (gamma) | 30–45 Hz |

For auditory oddball MEG data, power differences between conditions are expected
to be strongest over **bilateral temporal regions** (superior temporal gyrus, auditory
cortex), where the MMN generators are located. Unlike EEG — where the MMN appears
as a frontocentral negativity at Fz/FCz — MEG is sensitive to the tangential current
sources in the supratemporal plane, making temporal sensors the most informative here.

The `vlim='joint'` option links the colorbar scale across all five subplots,
making it easier to compare relative power across frequency bands.


In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────────
SHARED_COLORSCALE = True   # True: Standard and Deviant rows share the same colorscale
                           # False: each row scaled independently to its own maximum

# Plot spectral power topographies for both conditions and their difference.
#
# Important: X, Y, and Z channels for each sensor slot share the same 3D
# position — they are co-located by design (three orthogonal axes measured
# at the same physical point). plot_topomap() requires one unique spatial
# location per data point, so passing all three axes at once raises a
# 'overlapping positions' error.
#
# The fix is to use only one axis per topomap. We use Z (radial, normal to
# the scalp surface) as it typically has the highest SNR for MEG signals.
#
# EpochsSpectrum.plot_topomap() doesn't support a difference row, so we
# build the figure manually using mne.viz.plot_topomap().

import numpy as np

# Restrict to Z-axis mag channels only, excluding any bad channels.
bads = set(spectrum_deviant.info['bads'])
z_picks = [i for i, ch in enumerate(spectrum_deviant.ch_names)
           if ch.endswith(' Z') and not ch.startswith('BNC') and ch not in bads]
print(f'Using {len(z_picks)} Z-axis channels for topomaps')
if bads:
    print(f'Excluded bad channels: {[ch for ch in bads if ch.endswith(" Z")]}')

# Standard frequency bands
bands = [
    (0.5,  4,   'Delta\n(0.5–4 Hz)'),
    (4,    8,   'Theta\n(4–8 Hz)'),
    (8,    12,  'Alpha\n(8–12 Hz)'),
    (12,   30,  'Beta\n(12–30 Hz)'),
    (30,   45,  'Gamma\n(30–45 Hz)'),
]
n_bands = len(bands)

def mean_band_power(spectrum, picks, fmin, fmax):
    """Average power (linear) across epochs and across the frequency band.
    Returns shape (n_channels,) — one value per channel, for use in plot_topomap."""
    psds, freqs = spectrum.get_data(picks=picks, return_freqs=True)
    mask = (freqs >= fmin) & (freqs <= fmax)
    return psds[:, :, mask].mean(axis=(0, 2))  # (n_channels,)

# Compute per-band power for each condition
std_bands  = np.array([mean_band_power(spectrum_standard, z_picks, fmin, fmax)
                       for fmin, fmax, _ in bands])   # (n_bands, n_z_channels)
dev_bands  = np.array([mean_band_power(spectrum_deviant,  z_picks, fmin, fmax)
                       for fmin, fmax, _ in bands])
diff_bands = std_bands - dev_bands                    # positive = standard > deviant

# Build a single-axis info object for plot_topomap (Z channels only)
info_z = mne.pick_info(spectrum_deviant.info, z_picks)

# ── Colorscale ───────────────────────────────────────────────────────────────
# When SHARED_COLORSCALE=True, Standard and Deviant rows share a single vmax
# (the max across both), so their colors are directly comparable.
# The Difference row always uses its own diverging scale centered at zero.
# When False, each row scales independently to its own maximum.
if SHARED_COLORSCALE:
    shared_vmax = max(np.abs(std_bands).max(), np.abs(dev_bands).max())
else:
    shared_vmax = None

# ── Colormap notes ───────────────────────────────────────────────────────────
# Standard and Deviant rows use 'Reds' (white → red): higher power = deeper red.
# Difference row uses 'RdBu_r' (diverging, centered at zero):
#   Red  = standard > deviant  (more power in standard)
#   Blue = deviant > standard  (more power in deviant)

row_labels = ['Standard', 'Deviant', 'Difference\n(Std − Dev)']
row_data   = [std_bands, dev_bands, diff_bands]

fig, axes = plt.subplots(3, n_bands, figsize=(3.5 * n_bands, 9))

for row_idx, (row_label, data) in enumerate(zip(row_labels, row_data)):
    if row_idx == 2:
        # Difference row: always its own diverging scale
        vmax = np.abs(data).max()
        vmin = -vmax
        cmap = 'RdBu_r'
    else:
        # Standard / Deviant rows
        vmax = shared_vmax if SHARED_COLORSCALE else np.abs(data).max()
        vmin = 0
        cmap = 'Reds'

    for col_idx, (band_label, band_data) in enumerate(zip([b[2] for b in bands], data)):
        ax = axes[row_idx, col_idx]
        mne.viz.plot_topomap(
            band_data,
            info_z,
            axes=ax,
            show=False,
            cmap=cmap,
            vlim=(vmin, vmax),
            sensors=True,
            contours=4,
        )
        if row_idx == 0:
            ax.set_title(band_label, fontsize=10)
        if col_idx == 0:
            ax.set_ylabel(row_label, fontsize=10)

fig.suptitle('Spectral Power Topography — Z axis: Standard vs Deviant', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────────
SHARED_COLORSCALE = True   # True: Standard and Deviant rows share the same colorscale
                           # False: each row scaled independently to its own maximum

# Same three-row layout with custom frequency bands narrowed to the MMN-relevant range.
# Reuses z_picks, info_z, mean_band_power(), and row_labels from the cell above.
custom_bands = [
    (1,   4,  'Delta\n(1–4 Hz)'),
    (4,   8,  'Theta\n(4–8 Hz)'),
    (8,   12, 'Alpha\n(8–12 Hz)'),
    (12,  30, 'Beta\n(12–30 Hz)'),
]
n_bands_custom = len(custom_bands)

std_custom  = np.array([mean_band_power(spectrum_standard, z_picks, fmin, fmax)
                        for fmin, fmax, _ in custom_bands])
dev_custom  = np.array([mean_band_power(spectrum_deviant,  z_picks, fmin, fmax)
                        for fmin, fmax, _ in custom_bands])
diff_custom = std_custom - dev_custom

# ── Colorscale ───────────────────────────────────────────────────────────────
# When SHARED_COLORSCALE=True, Standard and Deviant rows share a single vmax
# (the max across both), so their colors are directly comparable.
# The Difference row always uses its own diverging scale centered at zero.
if SHARED_COLORSCALE:
    shared_vmax = max(np.abs(std_custom).max(), np.abs(dev_custom).max())
else:
    shared_vmax = None

# Colormap: Reds for standard/deviant rows, RdBu_r for the difference row.
# In the difference row: red = standard > deviant, blue = deviant > standard.
fig, axes = plt.subplots(3, n_bands_custom, figsize=(3.5 * n_bands_custom, 9))

for row_idx, (row_label, data) in enumerate(zip(row_labels, [std_custom, dev_custom, diff_custom])):
    if row_idx == 2:
        # Difference row: always its own diverging scale
        vmax = np.abs(data).max()
        vmin = -vmax
        cmap = 'RdBu_r'
    else:
        # Standard / Deviant rows
        vmax = shared_vmax if SHARED_COLORSCALE else np.abs(data).max()
        vmin = 0
        cmap = 'Reds'

    for col_idx, ((fmin, fmax, band_label), band_data) in enumerate(zip(custom_bands, data)):
        ax = axes[row_idx, col_idx]
        mne.viz.plot_topomap(
            band_data,
            info_z,
            axes=ax,
            show=False,
            cmap=cmap,
            vlim=(vmin, vmax),
            sensors=True,
            contours=4,
        )
        if row_idx == 0:
            ax.set_title(band_label, fontsize=10)
        if col_idx == 0:
            ax.set_ylabel(row_label, fontsize=10)

fig.suptitle('Spectral Power Topography — Z axis: Standard vs Deviant (Custom Bands)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Why does the difference map show blue patches if standard > deviant everywhere in the PSD plot?

This is a useful lesson in how global averages can mask spatial variability.

The Step 5 PSD overlay averages across **all channels and all epochs**, collapsing
everything to a single line per condition. Standard > deviant holds **on average**
across the whole sensor array. The difference topomap, however, shows the
difference **per channel individually** — and individual channels can go the other
way even if the global mean doesn't.

Blue patches in the difference map (deviant > standard locally) can arise from:

1. **Spatial heterogeneity of the deviant response.** The MMN suppression isn't
   uniform across the scalp. Some regions (e.g. frontal) may show deviant-related
   activity that locally *increases* power, while auditory/temporal regions show the
   expected suppression. A global average hides this spatial pattern entirely.

2. **Noisier deviant PSD estimates.** With only ~160 deviant trials vs ~840 standards,
   the per-channel deviant PSD has higher variance. A handful of slightly higher-amplitude
   epochs on a specific channel can push its deviant mean above the standard mean locally,
   even when the global average goes the other way.

3. **Residual artifacts.** If a small number of deviant epochs had higher-amplitude noise
   on specific channels that survived cleaning, those channels would show inflated deviant
   power.

> **What to look for:** If blue patches cluster consistently in a specific scalp region
> across multiple frequency bands, that is potentially meaningful neural signal (e.g.
> frontal deviant-related power increase). If they are spatially scattered and inconsistent
> across bands, it is more likely noise driven by the smaller deviant trial count.

## 7. Epoch Image Map — `plot_image()`

An **epoch image** is one of the most informative single-channel visualizations available.
It stacks all trials as rows of pixels:

- **X-axis:** time (within the epoch window)
- **Y-axis:** trial number (each row = one epoch, in acquisition order)
- **Color:** signal amplitude at that time × trial
- **Bottom panel:** mean ERP ± 95% bootstrapped confidence interval

This makes it easy to:
1. Spot **single-trial artifacts** (bright/dark horizontal streaks)
2. See whether the response is **consistent across trials** (a coherent vertical band)
3. Identify **trial-by-trial drift** (gradual color shift from top to bottom, indicating
   non-stationarity over the ~24 min recording)

### What to expect for the MMN in OPM MEG

The MMN is generated in **bilateral auditory cortex** (superior temporal gyrus and
Heschl's gyrus, in the supratemporal plane). In MEG, this manifests as a characteristic
**dipolar magnetic field pattern** over each temporal lobe — not the frontocentral
negativity seen in EEG, which reflects the same sources projected differently onto
the scalp surface. The channel we plot here (`T13 2Z Z`) sits over left temporal
cortex and should show this response directly.

**Temporal window:** The MMN peaks within **100–250 ms** after deviant onset. Several
factors influence where in that window the peak falls:

| Factor | Effect on MMN latency |
|--------|----------------------|
| Magnitude of deviance (frequency, duration, intensity) | Larger deviance → earlier, stronger MMN |
| ISI | Longer ISI → later MMN |
| Participant age | Older participants → longer latency |
| Attention state | Passive listening can shift latency relative to active tasks |

For this paradigm (200 Hz frequency shift, ~800 ms mean ISI, passive fixation), expect
the peak somewhere in the **130–200 ms** range.


Single-trial MMN visibility is limited — SNR is typically ~0.1–0.3 per trial — so
the response will be clearest in the mean ERP panel at the bottom of the image,
not in individual trial rows.

> **Tip:** The epoch window extends well past the tone offset (400 ms) to capture the
> full MMN and any later components. The ISI jitter (700–900 ms) means there is no
> stimulus-locked activity after ~900 ms post-onset, so the late portion of the epoch
> serves as a useful noise reference.


In [ ]:
# Pick one representative sensor to start with.
# You can change this to any channel name in epochs.ch_names.
# Check plot_sensors() above to identify channel locations.

example_channel = 'T13 2Z Z'  # left temporal — shows a clear MMN response
print(f'Plotting image for channel: {example_channel}')

fig = epochs[DEVIANT_COND].plot_image(
    picks=example_channel,
    sigma=0.5,          # light Gaussian smoothing across trials (reduces visual noise)
    title=f'Epoch Image — {example_channel} ({DEVIANT_COND})',
)


In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────────
SHARED_COLORSCALE = True   # True: same vmin/vmax across all 3 plots (fair comparison)
                           # False: MNE chooses independently per plot (maximises contrast)

# ── Equalized subsample (computed once, reused below) ────────────────────────
rng = np.random.default_rng(seed=42)
n_deviant  = len(epochs[DEVIANT_COND])
n_standard = len(epochs[STANDARD_COND])
subsample_idx  = rng.choice(n_standard, size=n_deviant, replace=False)
epochs_std_sub = epochs[STANDARD_COND][subsample_idx]

print(f'Deviant trials:               {n_deviant}')
print(f'Standard trials (original):   {n_standard}')
print(f'Standard trials (subsampled): {len(epochs_std_sub)}')

# ── Three figures ─────────────────────────────────────────────────────────────
# 1. Deviant — unequalized (all deviant trials, for reference)
# 2. Standard — unequalized (all standard trials; note the tighter CI vs deviant)
# 3. Standard — equalized (subsampled to match deviant N; CI should widen to match)
#
# Comparing figures 1 and 2 shows the CI difference due to unequal trial counts.
# Comparing figures 1 and 3 gives a fair comparison of response variability.

plots = [
    (epochs[DEVIANT_COND],  f'{DEVIANT_COND} — unequalized (n={n_deviant})'),
    (epochs[STANDARD_COND], f'{STANDARD_COND} — unequalized (n={n_standard})'),
    (epochs_std_sub,        f'{STANDARD_COND} — equalized (n={len(epochs_std_sub)})'),
]

# ── Shared colorscale ─────────────────────────────────────────────────────────
# When SHARED_COLORSCALE=True, compute vmin/vmax from the 99th percentile of
# absolute values pooled across all three epoch sets, then apply the same limits
# to every plot.  This makes amplitude differences visible across figures.
# When False, MNE chooses its own scale per plot — maximises within-plot contrast
# but makes cross-plot amplitude comparisons unreliable.
if SHARED_COLORSCALE:
    all_data = np.concatenate([
        epo.copy().pick(example_channel).get_data()
        for epo, _ in plots
    ], axis=0)
    vmax = float(np.percentile(np.abs(all_data), 99))
    vmin = -vmax
    print(f'Shared colorscale: vmin={vmin:.3e}, vmax={vmax:.3e} T')
else:
    vmin = vmax = None   # let MNE decide per plot
    print('Independent colorscale per plot (MNE default)')

for epo, title in plots:
    epo.plot_image(
        picks=example_channel,
        sigma=0.5,
        vmin=vmin,
        vmax=vmax,
        scalings=dict(mag=1.),
        title=f'{title}: {example_channel}',
        show=False,
    )
    plt.show()

> **Tip:** You can sort epochs by an external variable using the `order` parameter.
> For example, sorting by trial index (acquisition time) can reveal slow drift across the
> ~24 min recording session. Sorting by deviant position within a run (early vs late standards
> preceding the deviant) could reveal refractoriness effects:
> ```python
> # Sort deviant epochs by their original trial index
> trial_indices = epochs[DEVIANT_COND].selection  # original indices in the full events array
> sort_order = np.argsort(trial_indices)           # already sorted, but useful as a template
> epochs[DEVIANT_COND].plot_image(picks=example_channel, order=sort_order)
> ```

## 8. Topographic Image Map — `plot_topo_image_epochs()`

`plot_topo_image_epochs()` extends the image map concept to **all sensors simultaneously**,
rendering a miniature epoch image at each sensor's location on a head diagram.

This lets you see at a glance:
- Which **regions of the scalp** show the strongest response
- Whether the MMN signal shows the expected **bilateral temporal distribution** —
  in MEG, the response is maximal over auditory cortex on both sides, appearing as
  opposing magnetic field polarities on either side of each temporal lobe (a dipolar
  pattern). This is distinct from EEG, where the MMN appears frontocentrally at Fz.
- Whether any **spatial clusters** of sensors show anomalous patterns
  (e.g., sensors near the ears picking up muscle noise)
- How the **standard and deviant** responses differ across the whole sensor array

Since OPM data has a single channel type (unlike SQUID MEG which has separate
magnetometers and gradiometers), we don't need to filter by channel type or pass
a layout object — MNE will use the digitized sensor positions directly.

> **Practical note:** These tiny plots are best explored at full screen. In JupyterLab,
> right-click the figure → *Create New View for Output* to detach and resize it.
> In interactive Qt mode you can also click individual sensor images to pop open
> a full-sized `plot_image()` for that sensor.


In [ ]:
# First, drop any remaining outlier epochs that could dominate the colorscale.
# We use a simple peak-to-peak amplitude threshold.

# Compute peak-to-peak per epoch per channel
# get_data() has no exclude argument — pick first to drop bad channels,
# then call get_data().  Bad channels often have extreme amplitudes that
# would inflate the ptp threshold and cause good epochs to be incorrectly dropped.
data = epochs[DEVIANT_COND].copy().pick('mag', exclude='bads').get_data()  # (n_epochs, n_ch, n_times)
ptp  = np.ptp(data, axis=-1).max(axis=-1)            # max p-t-p across channels, per epoch

threshold_pT = np.percentile(ptp, 95)                # 95th percentile as threshold
good_mask    = ptp < threshold_pT

print(f'Keeping {good_mask.sum()} / {len(good_mask)} deviant epochs '
      f'(threshold: {threshold_pT*1e12:.1f} pT peak-to-peak)')

epochs_deviant_clean = epochs[DEVIANT_COND][good_mask]


In [ ]:
# plot_topo_image_epochs: one mini epoch-image at each sensor location.
# NOTE: do NOT use `from mne.viz import ...` here — that triggers a fresh import
# resolution which can pick up a different MNE installation (e.g. miniconda).
# Instead, access through the `mne` object already imported in Cell 2, which is
# bound to the correct uv environment.

plot_topo_image_epochs = mne.viz.plot_topo_image_epochs

# plot_topo_image_epochs has no picks argument — pre-select mag channels first.
# This is essential: non-mag channels (eyetracking, BNC, stim) have completely
# different amplitude scales and will ruin MNE's auto-colorscaling if included.
# exclude='bads' also removes any channels flagged in epochs.info['bads'].
epochs_mag = epochs_deviant_clean.copy().pick('mag', exclude='bads')

fig = plot_topo_image_epochs(
    epochs_mag,
    sigma=1.0,
    title=f'Topo Image — {DEVIANT_COND}',
    show=True,
)


## 9. Summary

Here's a quick reference for what each visualization is best suited for:

| Method | Best for |
|--------|----------|
| `epochs.plot()` | Interactive artifact inspection; marking bad epochs/channels |
| `epochs.plot(butterfly=True, group_by=axis_groups)` | Seeing global waveform shape per field axis (X/Y/Z); spotting axis-specific noise |
| `epochs.plot_sensors()` | Confirming 2D layout and 3D helmet geometry |
| `spectrum.plot()` | Verifying filter settings; checking 60 Hz line noise / sensor noise floor |
| `spectrum.plot_topomap()` | Spatial distribution of spectral power per frequency band |
| `epochs.plot_image()` | Single-channel trial-by-trial consistency; drift over ~24 min session |
| `plot_topo_image_epochs()` | Spatial overview of trial structure across all sensors |

### What to expect in this dataset

- **MMN window:** 100–250 ms post-onset (expect peak ~130–200 ms for this paradigm)
- **MMN distribution:** bilateral temporal in MEG (superior temporal gyrus / Heschl's
  gyrus); note this differs from EEG where the MMN appears frontocentrally at Fz
- **Standard vs deviant:** ~840 vs ~160 trials (after preprocessing rejection, somewhat fewer)
- **Tone offset:** 400 ms — any activity past this reflects post-stimulus processing, not
  the acoustic onset response
- **Blink trials** (green fixation, ≥1100 ms post-deviant) are separate events and
  do not appear in `deviant_onset` epochs

### Next steps
- Estimating evoked responses
- Time frequency analysis
- Source localization
